In [10]:
import pandas as pd

df = pd.read_csv("C:\\Users\\lolla\\Documents\\Github Projects\\New folder\\data\\telecom_churn.csv")

In [11]:
df.head()

,Churn,AccountWeeks,ContractRenewal,DataPlan,DataUsage,CustServCalls,DayMins,DayCalls,MonthlyCharge,OverageFee,RoamMins
0,0,128,1,1,2.7,1,265.1,110,89.0,9.87,10.0
1,0,107,1,1,3.7,1,161.6,123,82.0,9.78,13.7
2,0,137,1,0,0.0,0,243.4,114,52.0,6.06,12.2
3,0,84,0,0,0.0,2,299.4,71,57.0,3.10,6.6
4,0,75,0,0,0.0,3,166.7,113,41.0,7.42,10.1


In [12]:
df.shape
df.info()
df.isnull().sum()
df["Churn"].value_counts()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3333 entries, 0 to 3332
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Churn            3333 non-null   int64  
 1   AccountWeeks     3333 non-null   int64  
 2   ContractRenewal  3333 non-null   int64  
 3   DataPlan         3333 non-null   int64  
 4   DataUsage        3333 non-null   float64
 5   CustServCalls    3333 non-null   int64  
 6   DayMins          3333 non-null   float64
 7   DayCalls         3333 non-null   int64  
 8   MonthlyCharge    3333 non-null   float64
 9   OverageFee       3333 non-null   float64
 10  RoamMins         3333 non-null   float64
dtypes: float64(5), int64(6)
memory usage: 286.6 KB


Churn
0    2850
1     483
Name: count, dtype: int64

In [13]:
X = df.drop("Churn", axis=1)
y = df["Churn"]

In [14]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [15]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [16]:
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X.columns)

In [17]:
import joblib

joblib.dump(scaler, "../models/scaler.pkl")

['../models/scaler.pkl']

In [18]:
from xgboost import XGBClassifier

In [19]:
model = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    random_state=42,
    eval_metric="logloss"
)

model.fit(X_train_scaled, y_train)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'logloss'


In [20]:
y_pred = model.predict(X_test_scaled)
y_proba = model.predict_proba(X_test_scaled)[:, 1]

In [21]:
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.9280359820089955
ROC-AUC: 0.8454150841020077

Classification Report:
               precision    recall  f1-score   support

           0       0.94      0.98      0.96       570
           1       0.82      0.65      0.72        97

    accuracy                           0.93       667
   macro avg       0.88      0.81      0.84       667
weighted avg       0.92      0.93      0.92       667



In [22]:
import joblib

joblib.dump(model, "../models/churn_model.pkl")

['../models/churn_model.pkl']

In [29]:
cluster_features = df[[
    "AccountWeeks",
    "ContractRenewal",
    "DataPlan",
    "DataUsage",
    "CustServCalls",
    "DayMins",
    "DayCalls",
    "MonthlyCharge",
    "OverageFee",
    "RoamMins"
]]

In [30]:
from sklearn.preprocessing import StandardScaler

cluster_scaler = StandardScaler()
X_cluster = cluster_scaler.fit_transform(cluster_features)

In [31]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=4, random_state=42)
clusters = kmeans.fit_predict(X_cluster)

In [32]:
df["Cluster"] = clusters
df.head()

,Churn,AccountWeeks,ContractRenewal,DataPlan,DataUsage,CustServCalls,DayMins,DayCalls,MonthlyCharge,OverageFee,RoamMins,Cluster
0,0,128,1,1,2.7,1,265.1,110,89.0,9.87,10.0,2
1,0,107,1,1,3.7,1,161.6,123,82.0,9.78,13.7,2
2,0,137,1,0,0.0,0,243.4,114,52.0,6.06,12.2,1
3,0,84,0,0,0.0,2,299.4,71,57.0,3.10,6.6,0
4,0,75,0,0,0.0,3,166.7,113,41.0,7.42,10.1,0


In [33]:
df.groupby("Cluster").mean(numeric_only=True)

,Churn,AccountWeeks,ContractRenewal,DataPlan,DataUsage,CustServCalls,DayMins,DayCalls,MonthlyCharge,OverageFee,RoamMins
Cluster,,,,,,,,,,,
0,0.410828,104.194268,0.000000,0.264331,0.797420,1.449045,186.321338,100.605096,57.410510,10.175223,10.491083
1,0.180196,97.838537,1.000000,0.000000,0.071320,1.543265,215.964585,99.358608,56.450045,10.896432,10.013381
2,0.063030,100.871515,0.989091,1.000000,2.785830,1.545455,181.019636,99.631515,76.411030,10.172327,10.315758
3,0.093197,103.668220,1.000000,0.013048,0.086356,1.630009,139.094129,102.129543,40.371482,9.039627,10.336626


In [34]:
import joblib

joblib.dump(kmeans, "../models/kmeans.pkl")
joblib.dump(cluster_scaler, "../models/cluster_scaler.pkl")

['../models/cluster_scaler.pkl']

In [35]:
features = df[[
    "AccountWeeks",
    "ContractRenewal",
    "DataPlan",
    "DataUsage",
    "CustServCalls",
    "DayMins",
    "DayCalls",
    "MonthlyCharge",
    "OverageFee",
    "RoamMins"
]]

In [36]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(features)

In [37]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=4, random_state=42)
df["Cluster"] = kmeans.fit_predict(X_scaled)

In [38]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

X = X_scaled
y = df["Churn"]   # or your target column

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = RandomForestClassifier()
model.fit(X_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [39]:
import joblib

joblib.dump(model, "models/churn_model.pkl")
joblib.dump(scaler, "models/scaler.pkl")
joblib.dump(kmeans, "models/kmeans.pkl")

FileNotFoundError: [Errno 2] No such file or directory: 'models/churn_model.pkl'

In [40]:
import os

os.makedirs("models", exist_ok=True)

In [41]:
import os
import joblib

# create folder if not exists
os.makedirs("models", exist_ok=True)

joblib.dump(model, "models/churn_model.pkl")
joblib.dump(scaler, "models/scaler.pkl")
joblib.dump(kmeans, "models/kmeans.pkl")

['models/kmeans.pkl']